<a href="https://colab.research.google.com/github/arinamartynyuk/-4-9/blob/main/%D0%9B%D0%913_%D0%9C%D0%B0%D1%80%D1%82%D0%B8%D0%BD%D1%8E%D0%BA4_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота 3. Машинне навчання
## Регресія: прогнозування вартості нерухомості

**Мета роботи:** виконати первинний аналіз даних, побудувати кілька моделей регресії, оцінити їхню якість та виконати підбір гіперпараметрів для моделі Random Forest.

### Датасет
Можна використати один із двох способів завантаження даних:

1. Завантажити файл з Google Drive:
https://drive.google.com/file/d/1u_-3X05st6uvrgQB1qNLyE8a7QjTJjvd/view?usp=sharing

2. Завантажити датасет з Kaggle:
https://www.kaggle.com/datasets/prokshitha/home-value-insights

> Посилання та вміст датасету на зовнішніх платформах можуть змінюватися.

## 1. Імпорт бібліотек
Імпортуйте бібліотеки, необхідні для роботи з даними, візуалізації та числових обчислень.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 2. Завантаження даних
Завантажте файл двома способами.

### Варіант 1. Завантаження файлу з комп’ютера
Завантажте CSV-файл у середовище Colab.

In [ ]:
from google.colab import files
uploaded = files.upload()
filename = next(iter(uploaded))
df = pd.read_csv(filename)
print("Датасет завантажено:", filename)
display(df.head())

### Варіант 2. Завантаження датасету з Kaggle
Завантажте датасет безпосередньо з Kaggle та прочитайте CSV-файл у DataFrame `df`.

In [ ]:
# Завантаження з Google Drive (за посиланням із шаблону)
!pip -q install gdown
import gdown
file_id = "1u_-3X05st6uvrgQB1qNLyE8a7qjTJjvd"
gdown.download(f"https://drive.google.com/uc?id={file_id}", "housing.csv", quiet=False)
df = pd.read_csv("housing.csv")
print("Датасет завантажено")
display(df.head())

## 3. Опис ознак датасету

- **Square_Footage** — загальна житлова площа будинку.
- **Num_Bedrooms** — кількість спалень.
- **Num_Bathrooms** — кількість ванних кімнат.
- **Year_Built** — рік побудови будинку.
- **Lot_Size** — розмір земельної ділянки.
- **Garage_Size** — розмір гаража або кількість паркомісць.
- **Neighborhood_Quality** — оцінка якості району.
- **House_Price** — ціна будинку, цільова змінна.

## 4. Первинний аналіз даних

### Завдання 1. Перегляд даних
Виведіть перші рядки датасету та перегляньте його структуру.

In [ ]:
display(df.head())
print("Розмір датасету:", df.shape)
df.info()

### Завдання 2. Перевірка дублікатів
Визначте кількість дубльованих рядків.

In [ ]:
print("Кількість дублікатів:", df.duplicated().sum())

### Завдання 3. Перевірка пропущених значень
Визначте кількість пропущених значень у кожному стовпці.

In [ ]:
print(df.isnull().sum())

### Завдання 4. Описова статистика
Виведіть основні статистичні характеристики числових ознак.

In [ ]:
display(df.describe())

## 5. Аналіз окремих ознак

### Завдання 5. Ознака `Neighborhood_Quality`
Виведіть частоти значень ознаки `Neighborhood_Quality`.

In [ ]:
display(df["Neighborhood_Quality"].value_counts().sort_index())

### Завдання 6. Візуалізація `Neighborhood_Quality`
Побудуйте діаграму розподілу значень цієї ознаки.

In [ ]:
sns.countplot(data=df, x="Neighborhood_Quality")
plt.title("Розподіл Neighborhood_Quality")
plt.xlabel("Neighborhood_Quality")
plt.ylabel("Кількість")
plt.show()

### Завдання 7. Ознака `Num_Bathrooms`
Виведіть частоти значень кількості ванних кімнат.

In [ ]:
display(df["Num_Bathrooms"].value_counts().sort_index())

### Завдання 8. Візуалізація `Num_Bathrooms`
Побудуйте діаграму розподілу кількості ванних кімнат.

In [ ]:
sns.countplot(data=df, x="Num_Bathrooms")
plt.title("Розподіл кількості ванних кімнат")
plt.xlabel("Num_Bathrooms")
plt.ylabel("Кількість")
plt.show()

### Додаткове завдання
За потреби побудуйте попарні графіки числових ознак і коротко проаналізуйте можливі залежності.

In [ ]:
numeric_columns = df.select_dtypes(include=np.number).columns
sns.pairplot(df[numeric_columns], corner=True)
plt.show()

## 6. Кореляційний аналіз

### Завдання 9. Матриця кореляції
Побудуйте кореляційну матрицю для числових ознак та відобразіть її у вигляді теплової карти.

In [ ]:
corr_matrix = df.select_dtypes(include=np.number).corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Кореляційна матриця")
plt.show()

### Завдання 10. Кореляція з цільовою змінною
Виведіть кореляцію кожної ознаки з `House_Price`, відсортувавши значення за спаданням.

In [ ]:
correlations = df.select_dtypes(include=np.number).corr()["House_Price"].sort_values(ascending=False)
display(correlations)

## 7. Побудова моделей регресії

### Завдання 11. Імпорт інструментів Scikit-learn
Імпортуйте засоби для:
- поділу вибірки;
- масштабування ознак;
- побудови Linear Regression, Ridge, Lasso, Random Forest та Gradient Boosting;
- створення Pipeline;
- обчислення R², MAE та MSE.

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

### Завдання 12. Формування `X` і `y`
Відокремте ознаки від цільової змінної `House_Price`.

In [ ]:
y = df["House_Price"].copy()
X = df.drop(columns=["House_Price"]).copy()

# Заповнення пропусків
for column in X.columns:
    if X[column].dtype == "object":
        X[column] = X[column].fillna(X[column].mode()[0])
    else:
        X[column] = X[column].fillna(X[column].median())
y = y.fillna(y.median())

# Кодування категоріальних ознак
X = pd.get_dummies(X, drop_first=True)
print("X:", X.shape, "y:", y.shape)
display(X.head())

### Завдання 13. Поділ даних
Поділіть дані на навчальну і тестову вибірки у співвідношенні 80/20. Для відтворюваності використайте `random_state=42`.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Навчальна вибірка:", X_train.shape)
print("Тестова вибірка:", X_test.shape)

### Завдання 14. Створення моделей
Створіть п’ять моделей:
1. Linear Regression;
2. Ridge;
3. Lasso;
4. Random Forest Regressor;
5. Gradient Boosting Regressor.

Для лінійних моделей використайте `Pipeline` зі `StandardScaler`. Для деревоподібних моделей масштабування не потрібне.

In [ ]:
models = {
    "Linear Regression": Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())]),
    "Ridge": Pipeline([("scaler", StandardScaler()), ("model", Ridge())]),
    "Lasso": Pipeline([("scaler", StandardScaler()), ("model", Lasso(max_iter=10000))]),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}
print("Створено 5 моделей")

### Завдання 15. Навчання та оцінювання моделей
Для кожної моделі:
- виконайте навчання на тренувальній вибірці;
- отримайте прогноз для тестової вибірки;
- обчисліть `R²`, `MAE`, `MSE`;
- збережіть результати для подальшого порівняння.

In [ ]:
results = []
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    predictions[name] = y_pred
    results.append({"Model": name, "R2": r2_score(y_test, y_pred),
                    "MAE": mean_absolute_error(y_test, y_pred),
                    "MSE": mean_squared_error(y_test, y_pred)})
results_df = pd.DataFrame(results).sort_values("R2", ascending=False)
display(results_df)

### Завдання 16. Графічне порівняння прогнозів
Для кожної моделі побудуйте графік **фактичні значення vs прогнозовані значення**. Додайте лінію ідеального прогнозу.

In [ ]:
for name, y_pred in predictions.items():
    plt.figure(figsize=(8, 6))
    plt.scatter(y_test, y_pred, alpha=0.6)
    low = min(y_test.min(), y_pred.min())
    high = max(y_test.max(), y_pred.max())
    plt.plot([low, high], [low, high], "r--", label="Ідеальний прогноз")
    plt.xlabel("Фактичні значення")
    plt.ylabel("Прогнозовані значення")
    plt.title(f"Фактичні vs прогнозовані: {name}")
    plt.legend()
    plt.show()

## 8. Підбір гіперпараметрів Random Forest

### Завдання 17. GridSearchCV
Виконайте підбір оптимальних параметрів для `RandomForestRegressor` за допомогою `GridSearchCV`.

Дослідіть щонайменше такі параметри:
- `n_estimators`;
- `max_features`;
- `max_depth`;
- `min_samples_split`.

Використайте крос-валідацію та метрику `R²`.

In [ ]:
param_grid = {
    "n_estimators": [100, 200],
    "max_features": ["sqrt", 1.0],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5]
}
grid_search = GridSearchCV(RandomForestRegressor(random_state=42), param_grid,
                          cv=5, scoring="r2", n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)
print("Найкращі параметри:", grid_search.best_params_)
print("R2 крос-валідації:", grid_search.best_score_)

### Завдання 18. Оптимізована модель Random Forest
Створіть нову модель з найкращими параметрами, навчіть її та обчисліть `R²`, `MAE`, `MSE` на тестовій вибірці.

In [ ]:
best_rf = grid_search.best_estimator_
y_pred_best_rf = best_rf.predict(X_test)
best_r2 = r2_score(y_test, y_pred_best_rf)
best_mae = mean_absolute_error(y_test, y_pred_best_rf)
best_mse = mean_squared_error(y_test, y_pred_best_rf)
print("R2:", best_r2)
print("MAE:", best_mae)
print("MSE:", best_mse)

### Завдання 19. Порівняння Random Forest до і після оптимізації
Порівняйте значення `R²`, `MAE` і `MSE` для базової та оптимізованої моделей.

In [ ]:
y_pred_baseline_rf = predictions["Random Forest"]
comparison = pd.DataFrame({
    "Model": ["Random Forest (базова)", "Random Forest (оптимізована)"],
    "R2": [r2_score(y_test, y_pred_baseline_rf), best_r2],
    "MAE": [mean_absolute_error(y_test, y_pred_baseline_rf), best_mae],
    "MSE": [mean_squared_error(y_test, y_pred_baseline_rf), best_mse]
})
display(comparison)

baseline_r2 = r2_score(y_test, y_pred_baseline_rf)
baseline_mae = mean_absolute_error(y_test, y_pred_baseline_rf)
baseline_mse = mean_squared_error(y_test, y_pred_baseline_rf)
print("R2 до оптимізації:", baseline_r2, "| після:", best_r2)
print("MAE до оптимізації:", baseline_mae, "| після:", best_mae)
print("MSE до оптимізації:", baseline_mse, "| після:", best_mse)
if best_r2 > baseline_r2 and best_mae < baseline_mae:
    print("Оптимізація покращила якість моделі за R2 та MAE.")
else:
    print("Оптимізація не дала одночасного покращення всіх метрик; оцінюємо результат за сукупністю показників.")

## 9. Аналіз прогнозів лінійної регресії

### Завдання 21. Таблиця прогнозів
Для моделі Linear Regression створіть таблицю з трьома стовпцями:
- фактична ціна;
- прогнозована ціна;
- похибка у відсотках.

Виведіть 10 випадкових рядків.

In [ ]:
y_pred_linear = predictions["Linear Regression"]
prediction_table = pd.DataFrame({
    "Фактична ціна": y_test.values,
    "Прогнозована ціна": y_pred_linear,
    "Похибка (%)": np.abs(y_test.values-y_pred_linear) / np.maximum(np.abs(y_test.values), 1e-10) * 100
})
display(prediction_table.sample(min(10, len(prediction_table)), random_state=42))

best_model_name = results_df.iloc[0]["Model"]
best_model_r2 = results_df.iloc[0]["R2"]
best_model_mae = results_df.iloc[0]["MAE"]
best_model_mse = results_df.iloc[0]["MSE"]
print("У лабораторній роботі проведено первинний аналіз даних нерухомості та побудовано п’ять моделей регресії: Linear Regression, Ridge, Lasso, Random Forest і Gradient Boosting.")
print(f"Найкраща базова модель за R2: {best_model_name} (R2={best_model_r2:.4f}, MAE={best_model_mae:.4f}, MSE={best_model_mse:.4f}).")
print(f"Оптимізована Random Forest має R2={best_r2:.4f}, MAE={best_mae:.4f}, MSE={best_mse:.4f} на тестовій вибірці.")
print("Виконано підбір гіперпараметрів GridSearchCV з крос-валідацією. Якість прогнозування залежить від характеристик будинку, району та представленості даних.")